In [2]:
import re
import random
import tqdm
import pandas as pd
from collections import defaultdict

In [3]:
WORD_BREAK = '</w>'

In [ ]:
def load_data_csv(filename) -> tuple[list, list]:
    data = pd.read_csv(filename)
    data.dropna(inplace=True)
    print(f"Loaded {len(data)} rows")
    return data['en'].to_list(), data['fr'].to_list()

In [5]:
def load_data(filename, num_samples=10000):
    lines = open(filename, encoding='utf-8').read().strip().split('\n')

    en_texts = []
    fr_texts = []
    if isinstance(num_samples, float):
        assert 0. < num_samples <= 1., 'For float num_samples, it has to be in (0, 1]'
        num_samples = int(len(lines)*num_samples)
    
    assert isinstance(num_samples, int), f'num_samples is of wrong type: {type(num_samples)}'
    assert num_samples >= 0, 'num_samples can\'t be negative'

    for line in lines[:num_samples]:
        parts = line.split('\t')
        if len(parts) < 2: continue
        
        en_texts.append(parts[0])
        fr_texts.append(parts[1])
    
    return en_texts, fr_texts

In [6]:
def sample_from_file(filename, k):
    en_texts, fr_texts = load_data(filename, num_samples=1.)

    if isinstance(k, float):
        assert 0. < k < 1., 'For float num_samples, it has to be in (0, 1]'
        k = int(len(en_texts)*k)

    assert isinstance(k, int), f'num_samples is of wrong type: {type(k)}'
    assert k >= 0, 'num_samples can\'t be negative'

    en_texts = random.sample(en_texts, k)
    fr_texts = random.sample(fr_texts, k)

    return en_texts, fr_texts

In [7]:
def split_to_words(texts):
    p = re.compile(r'[^\W_]+(?:[\'\-][^\W_]+)*\'?')
    
    all_words = set()
    for text in texts:
        words = p.findall(text)
        
        all_words.update(words)
    return all_words

In [8]:
def get_all_symbols(words):
    all_simbols = set()
    
    for word in words:
        all_simbols.update(list(word))
    return all_simbols

In [9]:
def get_all_ngrams(texts, rules):
    all_ngramgs = set()
    
    words = split_to_words(texts)
    symbols = get_all_symbols(words)
    rules_as_sumbols = [''.join(rule) for rule in rules]

    all_ngramgs.update(symbols)
    all_ngramgs.update(rules_as_sumbols)
    all_ngramgs.add(WORD_BREAK)
    
    return list(all_ngramgs)

In [10]:
def count_words(texts):
    vocab = defaultdict(int)
    
    p = re.compile(r'[^\W_]+(?:[\'\-][^\W_]+)*\'?')

    for text in texts:
        words = p.findall(text)

        for word in words:
            vocab[word] += 1
    return vocab

In [11]:
def make_sparse_vocab(vocab: defaultdict):
    v_out = defaultdict(int)

    for word, freq in vocab.items():
        sparse_word = ' '.join(list(word)) + ' </w>'
        v_out[sparse_word] = freq

    return v_out

In [12]:
def pair_freq(vocab: defaultdict):
    pairs = defaultdict(int)

    for word, freq in vocab.items():
        tokens = word.split(' ')

        for i in range(len(tokens)-1):
            pairs[tokens[i], tokens[i+1]] += freq
    
    return pairs

In [13]:
def merge_vocab(v_in, pair):
    v_out = defaultdict(int)

    p = re.compile(r'(?<!\S)' + re.escape(' '.join(pair)) + r'(?!\S)')

    for word, freq in v_in.items():
        new_word = p.sub(''.join(pair), word)
        v_out[new_word] = freq
     
    return v_out

In [14]:
def create_vocab(texts, n=100):
    vocab = count_words(texts)
    vocab_sparse = make_sparse_vocab(vocab)

    popular_pairs = []

    for i in tqdm.tqdm(range(n)):
        pairs = pair_freq(vocab_sparse)
        popular_pair = max(pairs, key=pairs.get)
        popular_pairs.append(popular_pair)

        vocab_sparse = merge_vocab(vocab_sparse, popular_pair)

    return vocab, popular_pairs

In [15]:
# en_texts, fr_texts = sample_from_file('./fra.txt', k=0.75)

In [22]:
en_texts, fr_texts = load_data_csv('en-fr-1-mil.csv')

In [23]:
en_vocab, en_rules = create_vocab(en_texts, n=4_500)
fr_vocab, fr_rules = create_vocab(fr_texts, n=4_500)

TypeError: expected string or bytes-like object, got 'float'

In [ ]:
en_ngrams = get_all_ngrams(en_texts, en_rules)
fr_ngrams = get_all_ngrams(fr_texts, fr_rules)

In [ ]:
import pickle

with open('en-rules.pkl', 'wb') as file:
    pickle.dump(en_rules, file)

with open('fr-rules.pkl', 'wb') as file:
    pickle.dump(fr_rules, file)


In [ ]:
with open('en-rules.pkl', 'rb') as file:
    print(pickle.load(file)[-100])

with open('fr-rules.pkl', 'rb') as file:
    print(pickle.load(file)[-100])

('wa', 'sh</w>')
('c', 'ée</w>')


In [ ]:
with open('en-tokens.pkl', 'wb') as file:
    pickle.dump(en_ngrams, file)

with open('fr-tokens.pkl', 'wb') as file:
    pickle.dump(fr_ngrams, file)

In [ ]:
with open('en-tokens.pkl', 'rb') as file:
    print(pickle.load(file)[-100])

with open('fr-tokens.pkl', 'rb') as file:
    print(pickle.load(file)[-100])

watch
ins</w>
